# Dataset Loader Utility — NumPy Arrays for From-Scratch Learning

**Systems for AI — provided helper for PA1 (and reusable for other datasets)**

This notebook loads standard image-classification datasets as plain **NumPy arrays**, so you can
train models from scratch (no PyTorch/TensorFlow required). It is provided so you spend your time on
the network and the measurements, not on data-download plumbing.

It returns a clean contract every time:
`X_train, y_train, X_test, y_test` — pixels as float32 in [0,1], labels as int64.

**Reusable:** it handles MNIST by default, but the same call loads Fashion-MNIST or others by
changing the `name` argument. It tries several data sources in order and falls back gracefully, so
it works whether or not your environment has internet or a particular library.

## Configuration — set your data location here

Set `DATA_DIR` to the folder where your dataset files live. This is the ONE place to set the path;
every cell below uses it. If you downloaded MNIST (e.g. from Kaggle) and unzipped it, point this at
that folder — nesting and doubled extensions are handled automatically.

Leave it as `None` to let the loader auto-search common locations, then fall back to downloading.

In [ ]:
# ====== EDIT THIS ======
DATA_DIR = None          # e.g. "/home/gokhale/Documents/MNIST"  |  "./data"  |  "~/mnist"
DATASET  = "mnist"       # or "fashion_mnist"
# =======================
import os
DATA_DIR = os.path.expanduser(DATA_DIR) if DATA_DIR else None
print("DATASET =", DATASET)
print("DATA_DIR =", DATA_DIR if DATA_DIR else "(none — will auto-search, then try download)")

In [ ]:
import numpy as np, os, glob, gzip, struct

# WHAT ARE IDX FILES? MNIST ships in "IDX" binary format (not clickable images). Each file has a
# 4-byte magic number (2051=images, 2049=labels), then dimensions, then raw bytes. The four files:
#   train-images-* (60k imgs)  train-labels-* (60k)   t10k-images-* (10k test)  t10k-labels-* (10k)
# "idx3"=3-D (images), "idx1"=1-D (labels), "t10k"=test set. Downloads may unzip messily (nested
# folders, doubled extensions); the reader below handles that via recursive search + magic check.

def _read_idx(path):
    op = gzip.open if path.endswith(".gz") else open
    with op(path, "rb") as f:
        magic, = struct.unpack(">I", f.read(4))
        if magic == 2051:
            cnt, r, c = struct.unpack(">III", f.read(12))
            return np.frombuffer(f.read(), np.uint8).reshape(cnt, r * c)
        elif magic == 2049:
            cnt, = struct.unpack(">I", f.read(4))
            return np.frombuffer(f.read(), np.uint8)
        raise ValueError("%s: magic %d not IDX" % (path, magic))

def _find_idx(root, keywords):
    cands = []
    for p in glob.glob(os.path.join(root, "**"), recursive=True):
        if not os.path.isfile(p):
            continue
        if all(k in p.lower() for k in keywords):
            try:
                with open(p, "rb") as f:
                    if struct.unpack(">I", f.read(4))[0] in (2051, 2049):
                        cands.append(p)
            except Exception:
                pass
    return sorted(cands, key=len)[0] if cands else None

def _try_local_idx(name, data_dir):
    roles = {"tr_i": ["train", "images"], "tr_l": ["train", "labels"],
             "te_i": ["t10k", "images"],  "te_l": ["t10k", "labels"]}
    found = {r: _find_idx(data_dir, kw) for r, kw in roles.items()}
    missing = [r for r, p in found.items() if p is None]
    if missing:
        raise FileNotFoundError("IDX files not found in %s for %s" % (data_dir, missing))
    Xtr = _read_idx(found["tr_i"]).astype(np.float32) / 255.0
    ytr = _read_idx(found["tr_l"]).astype(np.int64)
    Xte = _read_idx(found["te_i"]).astype(np.float32) / 255.0
    yte = _read_idx(found["te_l"]).astype(np.int64)
    print("Loaded MNIST from local IDX files in", data_dir)
    return Xtr, ytr, Xte, yte

def _autodiscover_idx_dir():
    candidates = [".", "./data", "./MNIST", "./mnist",
                  os.path.expanduser("~/Documents/MNIST"), os.path.expanduser("~/mnist"),
                  os.path.expanduser("~/Downloads/MNIST")]
    for d in candidates:
        if not os.path.isdir(d):
            continue
        for p in glob.glob(os.path.join(d, "**"), recursive=True):
            if os.path.isfile(p) and "train" in p.lower() and "images" in p.lower():
                try:
                    with open(p, "rb") as f:
                        if struct.unpack(">I", f.read(4))[0] == 2051:
                            return d
                except Exception:
                    pass
    return None

def _try_openml(name):
    from sklearn.datasets import fetch_openml
    key = {"mnist": "mnist_784", "fashion_mnist": "Fashion-MNIST"}[name]
    ds = fetch_openml(key, version=1, as_frame=False, parser="auto")
    return ds.data.astype(np.float32) / 255.0, ds.target.astype(np.int64)

def _try_keras(name):
    from tensorflow.keras import datasets as kd
    loader = {"mnist": kd.mnist, "fashion_mnist": kd.fashion_mnist}[name]
    (Xtr, ytr), (Xte, yte) = loader.load_data()
    X = np.concatenate([Xtr, Xte]).reshape(-1, 28 * 28).astype(np.float32) / 255.0
    y = np.concatenate([ytr, yte]).astype(np.int64)
    return X, y

def _synthetic(name, n=2000, d=784, c=10, seed=0):
    rng = np.random.default_rng(seed)
    print("WARNING: SYNTHETIC data (no real source found). Results are meaningless — debug only.")
    return rng.random((n, d)).astype(np.float32), rng.integers(0, c, size=n).astype(np.int64)

def load_dataset(name="mnist", data_dir=None, test_frac=0.1428, seed=0, allow_synthetic=True):
    """Load a dataset as NumPy arrays -> X_train, y_train, X_test, y_test
    (X float32 in [0,1], y int64). Source order: local IDX (data_dir or auto-discovered) ->
    OpenML -> Keras -> synthetic. Pass data_dir to point at your unzipped download."""
    search_dir = data_dir or _autodiscover_idx_dir()
    if search_dir:
        try:
            return _try_local_idx(name, search_dir)
        except Exception as e:
            print("  local IDX in '%s' unavailable (%s)" % (search_dir, type(e).__name__))
    else:
        print("  no local IDX folder found (set DATA_DIR to use a downloaded dataset)")
    X = y = None
    for src in (_try_openml, _try_keras):
        try:
            X, y = src(name); print("Loaded '%s' via %s: X=%s" % (name, src.__name__, X.shape)); break
        except Exception as e:
            print("  %s unavailable (%s)" % (src.__name__, type(e).__name__))
    if X is None:
        if not allow_synthetic:
            raise RuntimeError("Could not load '%s' from any source." % name)
        X, y = _synthetic(name)
    rng = np.random.default_rng(seed); idx = rng.permutation(len(X)); X, y = X[idx], y[idx]
    k = int(round((1 - test_frac) * len(X)))
    return X[:k], y[:k], X[k:], y[k:]

def one_hot(y, num_classes=10):
    return np.eye(num_classes, dtype=np.float32)[y]

print("Loader ready. Set DATA_DIR in the config cell, then: load_dataset(DATASET, data_dir=DATA_DIR)")


## Load MNIST and sanity-check

The cell below loads MNIST and verifies the arrays look right. On your cloud/container environment
the OpenML or Keras path will fetch the real data. (If you are fully offline, it falls back to
synthetic data with a loud warning — that is only for debugging your pipeline, not for real results.)

In [ ]:
X_train, y_train, X_test, y_test = load_dataset(DATASET, data_dir=DATA_DIR)
print(f"X_train {X_train.shape}, y_train {y_train.shape}")
print(f"X_test  {X_test.shape},  y_test  {y_test.shape}")
print(f"pixel range [{X_train.min():.2f}, {X_train.max():.2f}]  (should be [0,1])")
print(f"classes present: {sorted(set(y_train.tolist()))}")
print(f"dataset memory: X_train = {X_train.nbytes/1e6:.0f} MB  (a first measurement!)")

# one-hot for the output layer
Y_train = one_hot(y_train, 10)
print(f"one-hot labels: {Y_train.shape}, each row sums to {Y_train[0].sum():.0f}")

## Visualize a few digits (optional check)

Confirms the pixels actually form recognizable digits — a good sanity check that loading worked.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"label: {y_train[i]}"); ax.axis("off")
plt.tight_layout(); plt.show()

## Reusing this for other datasets

The same call loads other datasets by changing `name`:

```python
X_train, y_train, X_test, y_test = load_dataset("fashion_mnist")   # 28x28, 10 clothing classes
```

To add a new dataset, extend the source helpers (`_try_openml`, `_try_keras`) with its key — the
rest of the pipeline (split, one-hot, the contract) stays the same. This is why the loader is a
reusable utility, not a one-off MNIST script.

**For your assignment:** you now have `X_train, y_train, X_test, y_test` as NumPy arrays. Build your
network, train it with your own backprop, and start measuring — that is where the real work begins.